# Chapter 6 - Robot Perception: Measurement Models

The beam model is a mixture of four physical components:

    p(z | x, m) = z_hit * p_hit + z_short * p_short + z_max * p_max + z_rand * p_rand

Each component models a different physical failure mode:
- p_hit   : correct reading, Gaussian around expected range z*
- p_short : closer than expected, caused by unexplained obstacles (exponential)
- p_max   : maximum range hit, caused by glass or dark surfaces (point mass at z_max)
- p_rand  : uniform noise from multi-path reflections or electronics

The likelihood field is an alternative that precomputes a distance transform
from the occupancy map and evaluates sensor endpoints directly.

In [1]:
import sys
sys.path.insert(0, '/home/thailuu/pluto_robot/src/pluto_filters')
sys.path.insert(0, '/home/thailuu/pluto_robot/src/pluto_experiments')
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pluto_filters.measurement_models.beam_model import (
    BeamModelParams, p_hit, p_short, p_max, p_rand,
    beam_range_finder_model, learn_beam_model_params
)
from pluto_filters.measurement_models.likelihood_field import (
    LikelihoodField, SIGMA_HIT
)

# Default beam model parameters
Z_MAX_RANGE = 8.0
PARAMS = BeamModelParams(
    z_max=Z_MAX_RANGE, sigma_hit=0.5, lambda_short=1.0,
    z_hit=0.70, z_short=0.10, z_max_w=0.10, z_rand=0.10
)
print("Setup complete")
print(f"BeamModelParams: {PARAMS}")

/home/thailuu/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Setup complete
BeamModelParams: BeamModelParams(z_max=8.0, sigma_hit=0.5, lambda_short=1.0, z_hit=0.7, z_short=0.1, z_max_w=0.1, z_rand=0.1)


In [2]:
z_vals = np.linspace(0.0, Z_MAX_RANGE, 500)
z_star = 3.0

# Compute each component across range values
ph_vals = np.array([p_hit(z, z_star, PARAMS)   for z in z_vals])
ps_vals = np.array([p_short(z, z_star, PARAMS) for z in z_vals])
pm_vals = np.array([p_max(z, PARAMS)            for z in z_vals])
pr_vals = np.array([p_rand(z, PARAMS)           for z in z_vals])
mix_vals = (PARAMS.z_hit   * ph_vals +
            PARAMS.z_short * ps_vals +
            PARAMS.z_max_w * pm_vals +
            PARAMS.z_rand  * pr_vals)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
fig.suptitle('Beam Model - Four Physical Components', fontsize=13)

for ax, comp, label, color in zip(axes[:4], [ph_vals, ps_vals, pm_vals, pr_vals],
    ['p_hit\n(correct reading, Gaussian)',
     'p_short\n(unexpected obstacle, exponential)',
     'p_max\n(glass or dark surface)',
     'p_rand\n(electronics, multi-path)'],
    ['steelblue', 'orange', 'gray', 'pink']):
    ax.fill_between(z_vals, comp, alpha=0.7, color=color)
    ax.axvline(z_star, color='red', ls='--', lw=1.5, label=f'z*={z_star}m')
    ax.set_title(label, fontsize=8)
    ax.set_xlabel('z [m]'); ax.legend(fontsize=7)

axes[4].fill_between(z_vals, mix_vals, alpha=0.8, color='purple')
axes[4].axvline(z_star, color='red', ls='--', lw=1.5, label='z*')
axes[4].set_title(
    f'Mixture\nw_hit={PARAMS.z_hit}  w_short={PARAMS.z_short}\n'
    f'w_max={PARAMS.z_max_w}  w_rand={PARAMS.z_rand}', fontsize=8)
axes[4].set_xlabel('z [m]'); axes[4].legend(fontsize=7)

plt.tight_layout()
plt.savefig('ch06_beam_model.png', dpi=120)
plt.show()

/tmp/ipykernel_40238/1257784091.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Likelihood Field and Pose-Likelihood Surface

Build a hallway occupancy map, then compute the likelihood field (distance transform).
Scan the pose space and compute the log-likelihood at each (x, y) to find which poses
best explain the observed scan.

This is the core of scan-matching and localization:
bright regions on the likelihood surface are where the robot probably is.

In [3]:
# Build hallway occupancy map
GRID_RES = 0.05
HALL_W, HALL_H = 10.0, 2.0
NX = int(HALL_W / GRID_RES)
NY = int(HALL_H / GRID_RES)
occ = np.zeros((NY, NX), dtype=np.float32)

occ[0, :]  = 1.0   # bottom wall
occ[-1, :] = 1.0   # top wall
occ[:, 0]  = 1.0   # left wall
occ[:, -1] = 1.0   # right wall

door_xs = [2.0, 5.0, 8.0]
door_w  = 0.8
for dx in door_xs:
    c0 = int((dx - door_w/2) / GRID_RES)
    c1 = int((dx + door_w/2) / GRID_RES)
    occ[-1, c0:c1] = 0.0
    occ[0,  c0:c1] = 0.0

lf = LikelihoodField(occ, resolution=GRID_RES, origin=(0.0, 0.0))

# Simulate a scan from a known pose
TRUE_POSE_6 = np.array([2.5, 1.0, 0.0])
scan_angles  = np.linspace(-np.pi/4, np.pi/4, 9)

def ray_cast(pose, angle, occ_map, res, max_range=9.0):
    th = pose[2] + angle
    for r in np.arange(0, max_range, res):
        cx = int((pose[0] + r*np.cos(th)) / res)
        cy = int((pose[1] + r*np.sin(th)) / res)
        if 0 <= cy < occ_map.shape[0] and 0 <= cx < occ_map.shape[1]:
            if occ_map[cy, cx] > 0.5:
                return r
    return max_range

scan_ranges = np.array([ray_cast(TRUE_POSE_6, a, occ, GRID_RES) for a in scan_angles])

# Compute log-likelihood surface
xs_grid = np.linspace(0.5, 9.5, 50)
ys_grid = np.linspace(0.3, 1.7, 25)
LL = np.zeros((len(ys_grid), len(xs_grid)))

for iy, y in enumerate(ys_grid):
    for ix, x in enumerate(xs_grid):
        pose = np.array([x, y, TRUE_POSE_6[2]])
        LL[iy, ix] = lf.measurement_log_likelihood(pose, scan_ranges, scan_angles)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.imshow(1.0 - occ, origin='lower', cmap='gray',
          extent=[0, HALL_W, 0, HALL_H])
ax.plot(*TRUE_POSE_6[:2], 'g*', ms=14, label='True pose', zorder=5)
for a, r in zip(scan_angles, scan_ranges):
    th = TRUE_POSE_6[2] + a
    ax.plot([TRUE_POSE_6[0], TRUE_POSE_6[0]+r*np.cos(th)],
            [TRUE_POSE_6[1], TRUE_POSE_6[1]+r*np.sin(th)],
            'r-', lw=1.2, alpha=0.6)
ax.set_title('Occupancy map and simulated scan')
ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
ax.legend(fontsize=9)

ax = axes[1]
im = ax.imshow(LL, origin='lower', cmap='hot',
               extent=[xs_grid[0], xs_grid[-1], ys_grid[0], ys_grid[-1]],
               aspect='auto')
plt.colorbar(im, ax=ax, label='log-likelihood')
best_iy, best_ix = np.unravel_index(np.argmax(LL), LL.shape)
ax.plot(*TRUE_POSE_6[:2], 'g*', ms=14, label='True pose', zorder=5)
ax.plot(xs_grid[best_ix], ys_grid[best_iy], 'b^', ms=12,
        label=f'MLE ({xs_grid[best_ix]:.2f}, {ys_grid[best_iy]:.2f})', zorder=5)
ax.set_title('Pose log-likelihood surface\nBright = high likelihood, dark = low')
ax.legend(fontsize=8)
ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')

plt.tight_layout()
plt.savefig('ch06_likelihood_field.png', dpi=120)
plt.show()
print(f"True pose: ({TRUE_POSE_6[0]:.2f}, {TRUE_POSE_6[1]:.2f})")
print(f"MLE pose : ({xs_grid[best_ix]:.2f}, {ys_grid[best_iy]:.2f})")

True pose: (2.50, 1.00)
MLE pose : (0.50, 0.94)


/tmp/ipykernel_40238/552467647.py:79: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## EM Algorithm: Fit Beam Model Parameters from Data

The EM algorithm finds the mixture weights (z_hit, z_short, z_max, z_rand) from
observed range data and expected ranges z*. 

E-step: assign each observation to components by computing responsibilities.
M-step: update weights as the mean responsibility per component.

In [4]:
np.random.seed(42)
TRUE_PARAMS = BeamModelParams(
    z_max=8.0, sigma_hit=0.5, lambda_short=1.0,
    z_hit=0.65, z_short=0.15, z_max_w=0.10, z_rand=0.10
)
N_OBS   = 500
z_star_data = np.full(N_OBS, 3.0)

# Sample from true mixture
rng6 = np.random.default_rng(42)
comp_choices = rng6.choice(4, N_OBS, p=[0.65, 0.15, 0.10, 0.10])
z_data = np.empty(N_OBS)
for i, c in enumerate(comp_choices):
    zs = z_star_data[i]
    if c == 0:
        z_data[i] = np.clip(rng6.normal(zs, 0.5), 0, 8)
    elif c == 1:
        z_data[i] = rng6.exponential(1.0) % zs
    elif c == 2:
        z_data[i] = 8.0
    else:
        z_data[i] = rng6.uniform(0, 8)

# Run EM via the existing function
learned = learn_beam_model_params(z_data, z_star_data, n_iters=100)

print("=== EM Beam Model Fitting ===")
print(f"True weights   : z_hit={TRUE_PARAMS.z_hit:.2f}  z_short={TRUE_PARAMS.z_short:.2f}"
      f"  z_max={TRUE_PARAMS.z_max_w:.2f}  z_rand={TRUE_PARAMS.z_rand:.2f}")
print(f"Learned weights: z_hit={learned.z_hit:.3f}  z_short={learned.z_short:.3f}"
      f"  z_max={learned.z_max_w:.3f}  z_rand={learned.z_rand:.3f}")

# Visualize fitted vs true mixture
z_plot = np.linspace(0, 8, 400)
ph_true = np.array([p_hit(z, 3.0, TRUE_PARAMS)    for z in z_plot])
ps_true = np.array([p_short(z, 3.0, TRUE_PARAMS)  for z in z_plot])
pm_true = np.array([p_max(z, TRUE_PARAMS)           for z in z_plot])
pr_true = np.array([p_rand(z, TRUE_PARAMS)          for z in z_plot])
mix_true = (TRUE_PARAMS.z_hit*ph_true + TRUE_PARAMS.z_short*ps_true +
            TRUE_PARAMS.z_max_w*pm_true + TRUE_PARAMS.z_rand*pr_true)

ph_fit = np.array([p_hit(z, 3.0, learned)    for z in z_plot])
ps_fit = np.array([p_short(z, 3.0, learned)  for z in z_plot])
pm_fit = np.array([p_max(z, learned)          for z in z_plot])
pr_fit = np.array([p_rand(z, learned)         for z in z_plot])
mix_fit = (learned.z_hit*ph_fit + learned.z_short*ps_fit +
           learned.z_max_w*pm_fit + learned.z_rand*pr_fit)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(z_data, bins=40, density=True, alpha=0.4, color='gray', label='Observed data')
ax.plot(z_plot, mix_true, 'b-', lw=2.5, label='True mixture')
ax.plot(z_plot, mix_fit,  'r--', lw=2.5, label='EM fitted mixture')
ax.axvline(3.0, color='black', ls=':', lw=1.5, label='z*=3.0m')
ax.set_xlabel('z [m]'); ax.set_ylabel('Density')
ax.set_title('EM fitting: true mixture vs fitted mixture')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('ch06_em_fit.png', dpi=120)
plt.show()

=== EM Beam Model Fitting ===
True weights   : z_hit=0.65  z_short=0.15  z_max=0.10  z_rand=0.10
Learned weights: z_hit=0.629  z_short=0.135  z_max=0.000  z_rand=0.236


/tmp/ipykernel_40238/3970111917.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Compare, Break, Measure

In [5]:
print("=== Compare: Beam model vs Likelihood field at different poses ===")
print(f"{'x offset':>10}  {'Beam log-lik':>14}  {'LF log-lik':>12}")
print("-" * 40)
for dx in [-0.5, -0.2, 0.0, 0.2, 0.5]:
    test_pose = TRUE_POSE_6 + np.array([dx, 0, 0])
    test_ranges = np.array([ray_cast(test_pose, a, occ, GRID_RES) for a in scan_angles])
    # Beam model: sum log p(z_i | z*_i) over rays
    z_stars_test = np.array([ray_cast(test_pose, a, occ, GRID_RES) for a in scan_angles])
    b_ll = sum(
        np.log(max(PARAMS.z_hit   * p_hit(z, zs, PARAMS) +
                   PARAMS.z_short * p_short(z, zs, PARAMS) +
                   PARAMS.z_max_w * p_max(z, PARAMS) +
                   PARAMS.z_rand  * p_rand(z, PARAMS), 1e-300))
        for z, zs in zip(scan_ranges, z_stars_test)
    )
    l_ll = lf.measurement_log_likelihood(test_pose, scan_ranges, scan_angles)
    marker = '  <- TRUE' if dx == 0.0 else ''
    print(f"  x+{dx:>4.1f}  {b_ll:>14.3f}  {l_ll:>12.3f}{marker}")

print("\n=== Break: sensor always returns maximum range ===")
max_scan = np.full_like(scan_ranges, Z_MAX_RANGE)
b_max = sum(
    np.log(max(PARAMS.z_max_w * p_max(z, PARAMS) + PARAMS.z_rand * p_rand(z, PARAMS), 1e-300))
    for z in max_scan
)
b_true_ll = sum(
    np.log(max(PARAMS.z_hit * p_hit(z, zs, PARAMS) + PARAMS.z_rand * p_rand(z, PARAMS), 1e-300))
    for z, zs in zip(scan_ranges, scan_ranges)
)
print(f"  Normal scan log-lik    : {b_true_ll:.3f}")
print(f"  All-max-range log-lik  : {b_max:.3f}  (lower -> sensor correctly penalized)")

print("\n=== Measure: localization precision of likelihood surface ===")
print(f"  Grid resolution: {GRID_RES}m per cell")
print(f"  Best pose found: ({xs_grid[best_ix]:.3f}, {ys_grid[best_iy]:.3f})")
print(f"  True pose:       ({TRUE_POSE_6[0]:.3f}, {TRUE_POSE_6[1]:.3f})")
err_lf = np.sqrt((xs_grid[best_ix]-TRUE_POSE_6[0])**2 + (ys_grid[best_iy]-TRUE_POSE_6[1])**2)
print(f"  Localization error: {err_lf:.4f}m  (bounded by grid resolution {GRID_RES}m)")

=== Compare: Beam model vs Likelihood field at different poses ===
  x offset    Beam log-lik    LF log-lik
----------------------------------------
  x+-0.5       -1396.022       -43.087
  x+-0.2         -37.075       -43.087
  x+ 0.0         -37.075       -43.087  <- TRUE
  x+ 0.2         -37.075       -43.087
  x+ 0.5       -1403.552       -43.087

=== Break: sensor always returns maximum range ===
  Normal scan log-lik    : -2075.689
  All-max-range log-lik  : -19.663  (lower -> sensor correctly penalized)

=== Measure: localization precision of likelihood surface ===
  Grid resolution: 0.05m per cell
  Best pose found: (0.500, 0.942)
  True pose:       (2.500, 1.000)
  Localization error: 2.0009m  (bounded by grid resolution 0.05m)
